In [ ]:
import ydf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("1 - Project Data.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['Count'].unique()

In [ ]:
df['Churn Label'] = np.where(df['Churn Label'] == "Yes", 1, 0)


In [ ]:
df[df['Churn Label'] != df['Churn Value']]

In [ ]:
df.drop(columns=['Churn Label'], inplace=True)

In [ ]:
df.loc[2235]['Total Charges']


In [ ]:
df.loc[2236]['Total Charges']


In [ ]:
df.loc[2237]['Total Charges']

In [ ]:
df[df['CustomerID'] == "4472-LVYGI"]

In [ ]:
df.columns

In [ ]:
cols_for_numb = ['Count', 'Zip Code', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Paperless Billing',
       'Monthly Charges', 'Total Charges', 'Churn Value']

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.to_list()

In [ ]:
numerical_cols

In [ ]:
cols_to_switch = [col for col in cols_for_numb if col not in numerical_cols]

In [ ]:
cols_to_switch

In [ ]:
df_switch_numb = df[cols_to_switch]

In [ ]:
df_switch_numb.head()

In [ ]:
other_cols_list = ["Gender", "Total Charges"]

In [ ]:
col_yes_no = [col for col in df_switch_numb.columns.to_list() if col not in other_cols_list]

In [ ]:
for col in col_yes_no:
    print(df[col].unique())

In [ ]:
for col in col_yes_no:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0})

In [ ]:
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce').astype('float')
df['Total Charges'].fillna(0, inplace=True)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df[df['CustomerID'] == "4472-LVYGI"]

In [ ]:
print(df['Churn Reason'].nunique())
df['Churn Reason'].unique()

In [ ]:
df_non_numeric = df.select_dtypes(exclude='number')

In [ ]:
df_non_numeric.head()

In [ ]:
for col in df_non_numeric.columns.to_list():
    print(f"{col}: {df_non_numeric[col].unique()}\n\n   Count:  {df_non_numeric[col].nunique()}\n\n") 

In [ ]:
cols_to_drop = ["CustomerID", "Country", "State", "City", "Lat Long", "Churn Reason"]
df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
dummies_list = ["Internet Service", "Contract", "Payment Method"]
df = pd.get_dummies(data=df, columns=dummies_list, prefix=["IS", "Cnt", "PM"], dtype=int, drop_first=True)

In [ ]:
df.head()

In [ ]:
df.columns

`['Count', 'Zip Code', 'Latitude', 'Longitude', 'Total Charges', 'Churn Value']`

In [ ]:
df.info()

In [ ]:
correlation_matrix = df.corr(numeric_only=True)
plt.figure(figsize=(30,5))
sns.heatmap(data=correlation_matrix.loc[['Churn Value']], annot=True, center=0)

In [ ]:
correlation_matrix = df.corr(numeric_only=True)
plt.figure(figsize=(30,30))
sns.heatmap(data=correlation_matrix, annot=True)

In [ ]:
col_yes_no

In [ ]:
service_list = ['Phone Service',
 'Multiple Lines',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies']

df["Service Count"] = (df[service_list] == 1).sum(axis=1)

In [ ]:
df.info()

In [ ]:
correlation_matrix = df.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(30,5))
sns.heatmap(data=correlation_matrix.loc[['Churn Value']], annot=True, center=0)

In [ ]:
features = list(df.columns)
features.remove('Churn Value')

y = df['Churn Value']
X = df[features]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=1204, stratify=y)

X_train['Churn Value'] = y_train # This is adding the label/target back to X train
X_test['Churn Value'] = y_test   # Same for X_test

In [ ]:
tuner = ydf.RandomSearchTuner(num_trials=50)

In [ ]:
model = ydf.GradientBoostedTreesLearner(tuner=tuner, label='Churn Value').train(X_train)

In [ ]:
model.evaluate(X_train)

In [ ]:
model.evaluate(X_test)

In [ ]:
model.plot_tree(max_depth=3)

In [ ]:
model.describe()